<a href="https://colab.research.google.com/github/HSA2005-cloud/BIS-Lab/blob/main/5.%20Grey%20Wolf%20Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np

# ---------------------------------------
# 1. INITIALIZE PARAMETERS
# ---------------------------------------

N = 25
MAX_ITER = 100

BATTERY_CAPACITY = 100
MAX_CHARGE = 20
MAX_DISCHARGE = 20

INITIAL_SOC = 50
MIN_SOC = 10
MAX_SOC = 100

CHARGE_EFFICIENCY = 0.95
DISCHARGE_EFFICIENCY = 0.95

# ---------------------------------------
# 2. HOURLY ENERGY DATA (24 HOURS)
# ---------------------------------------

solar = np.array([
    0, 0, 0, 0, 0, 5,
    15, 30, 45, 55, 65, 70,
    75, 70, 60, 45, 25, 10,
    0, 0, 0, 0, 0, 0
])

wind = np.array([
    15, 15, 20, 20, 15, 15,
    10, 10, 10, 15, 20, 20,
    15, 15, 20, 25, 30, 35,
    30, 25, 20, 20, 15, 15
])

demand = np.array([
    35, 30, 30, 30, 35, 40,
    45, 50, 55, 60, 65, 65,
    60, 60, 65, 70, 75, 80,
    85, 80, 70, 60, 50, 40
])

grid_price = np.array([
    5, 5, 5, 5, 5, 5,
    6, 6, 6, 6, 7, 7,
    7, 7, 7, 8, 8, 10,
    10, 10, 8, 7, 6, 5
])

renewable = solar + wind

# ---------------------------------------
# 3. FITNESS FUNCTION
# ---------------------------------------

def evaluate(schedule):

    soc = INITIAL_SOC
    total_grid_cost = 0
    total_curtailment = 0
    total_grid = 0

    for t in range(24):

        action = schedule[t]

        if action < 0:

            # Battery charging
            charge = min(
                abs(action),
                MAX_CHARGE,
                (MAX_SOC - soc) / CHARGE_EFFICIENCY
            )

            soc += charge * CHARGE_EFFICIENCY

            # Use renewable surplus first
            surplus = max(0, renewable[t] - demand[t])

            grid = max(0, charge - surplus)
            curtailment = max(0, surplus - charge)

        else:

            # Battery discharging
            available = max(0, soc - MIN_SOC)

            discharge = min(
                action,
                MAX_DISCHARGE,
                available * DISCHARGE_EFFICIENCY,
                max(0, demand[t] - renewable[t])
            )

            soc -= discharge / DISCHARGE_EFFICIENCY

            grid = max(
                0,
                demand[t] - renewable[t] - discharge
            )

            curtailment = 0

        total_grid += grid
        total_grid_cost += grid * grid_price[t]
        total_curtailment += curtailment

    # Encourage renewable utilization
    # and discourage ending with a depleted battery

    fitness = (
        total_grid_cost
        + total_curtailment * 1
        + max(0, INITIAL_SOC - soc) * 50
    )

    return fitness


# ---------------------------------------
# 4. INITIALIZE WOLF POPULATION
# ---------------------------------------

wolves = np.random.uniform(
    -MAX_CHARGE,
    MAX_DISCHARGE,
    (N, 24)
)

# ---------------------------------------
# 5. GREY WOLF OPTIMIZATION
# ---------------------------------------

best_cost = float("inf")
best_schedule = None

for iteration in range(MAX_ITER):

    fitness = np.array([
        evaluate(wolf) for wolf in wolves
    ])

    # Identify Alpha, Beta and Delta
    sorted_indices = np.argsort(fitness)

    alpha = wolves[sorted_indices[0]].copy()
    beta = wolves[sorted_indices[1]].copy()
    delta = wolves[sorted_indices[2]].copy()

    # Store the best solution
    if fitness[sorted_indices[0]] < best_cost:
        best_cost = fitness[sorted_indices[0]]
        best_schedule = alpha.copy()

    # Adaptive parameter
    a = 2 - 2 * iteration / MAX_ITER

    # Update positions of all wolves
    for i in range(N):

        X = wolves[i]

        # Alpha update
        r1, r2 = np.random.rand(2, 24)
        A1 = 2 * a * r1 - a
        C1 = 2 * r2
        D_alpha = abs(C1 * alpha - X)
        X1 = alpha - A1 * D_alpha

        # Beta update
        r1, r2 = np.random.rand(2, 24)
        A2 = 2 * a * r1 - a
        C2 = 2 * r2
        D_beta = abs(C2 * beta - X)
        X2 = beta - A2 * D_beta

        # Delta update
        r1, r2 = np.random.rand(2, 24)
        A3 = 2 * a * r1 - a
        C3 = 2 * r2
        D_delta = abs(C3 * delta - X)
        X3 = delta - A3 * D_delta

        # Final position update
        new_position = (X1 + X2 + X3) / 3

        # Apply power constraints
        wolves[i] = np.clip(
            new_position,
            -MAX_CHARGE,
            MAX_DISCHARGE
        )

# ---------------------------------------
# 6. DISPLAY RESULTS
# ---------------------------------------

print("\nGWO PARAMETERS")
print("-----------------------")
print("Number of Wolves:", N)
print("Maximum Iterations:", MAX_ITER)
print("Battery Capacity:", BATTERY_CAPACITY, "kWh")
print("Maximum Charge:", MAX_CHARGE, "kW")
print("Maximum Discharge:", MAX_DISCHARGE, "kW")
print("Initial SOC:", INITIAL_SOC, "kWh")
print("Minimum SOC:", MIN_SOC, "kWh")
print("Maximum SOC:", MAX_SOC, "kWh")

print("\nOPTIMIZATION RESULTS")
print("-----------------------")
print("Best Operating Cost:", round(best_cost, 2))

print("\nOPTIMAL BATTERY SCHEDULE")
print("Hour       Battery Action")

for hour, action in enumerate(best_schedule):
    if action < 0:
        print(hour, "       Charge:", round(abs(action), 2), "kW")
    else:
        print(hour, "       Discharge:", round(action, 2), "kW")

print("\nTotal Renewable Generation:",
      np.sum(renewable), "kWh")

print("Total Electricity Demand:",
      np.sum(demand), "kWh")



GWO PARAMETERS
-----------------------
Number of Wolves: 25
Maximum Iterations: 100
Battery Capacity: 100 kWh
Maximum Charge: 20 kW
Maximum Discharge: 20 kW
Initial SOC: 50 kWh
Minimum SOC: 10 kWh
Maximum SOC: 100 kWh

OPTIMIZATION RESULTS
-----------------------
Best Operating Cost: 29.01

OPTIMAL BATTERY SCHEDULE
Hour       Battery Action
0        Charge: 0.28 kW
1        Charge: 0.55 kW
2        Charge: 0.3 kW
3        Charge: 0.13 kW
4        Charge: 0.03 kW
5        Charge: 0.63 kW
6        Charge: 0.41 kW
7        Charge: 0.32 kW
8        Discharge: 8.26 kW
9        Charge: 9.98 kW
10        Discharge: 1.18 kW
11        Charge: 20.0 kW
12        Charge: 20.0 kW
13        Discharge: 7.52 kW
14        Discharge: 1.88 kW
15        Discharge: 2.75 kW
16        Charge: 19.29 kW
17        Charge: 1.7 kW
18        Charge: 11.31 kW
19        Charge: 8.7 kW
20        Charge: 14.42 kW
21        Charge: 7.87 kW
22        Charge: 0.74 kW
23        Charge: 1.64 kW

Total Renewable Generation